In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.imputation import load_time_indexed_csv, impute_loads_by_gap_categories_safe, apply_imputed_columns, missing_summary
from src.time_features import add_calendar_features, get_calendar_feature_columns
from src.lag_features import add_lag_features, add_rolling_features, add_trend_features

import src.dataset_builder as dataset_builder
import src.profile_features as profile_features

### Change path according to sample reso.

In [41]:
INPUT_PATH = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\resampled\LF_data_droped_15m.csv"
SAVE_DIR = Path(r"C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min") # chaneg path when resolution is diff.

In [42]:
df = load_time_indexed_csv(INPUT_PATH)
print(df.shape)
print(df.columns)

(11232, 9)
Index(['BA_Soc', 'BU_TotActPwr_Academy', 'BA_TotActPwr_BESS_AC_Panel1',
       'BA_TotActPwr_BESS_AC_Panel2', 'BU_TotActPwr_SDB_EL_Substation',
       'BU_TotActPwr_Tech_Room', 'PV_WS_AirTemp', 'PV_WS_Radiation',
       'PV_WS_RelHum'],
      dtype='str')


In [43]:
impute_cols = [
    "BA_Soc",
    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_SDB_EL_Substation",
    "BU_TotActPwr_Tech_Room",
    "PV_WS_AirTemp",
    "PV_WS_Radiation",
    "PV_WS_RelHum",
]

df_imputed_only, impute_report = impute_loads_by_gap_categories_safe(
    df,
    impute_cols,
    freq_minutes=15,
    short_gap_hours=2.0,
    medium_gap_hours=24.0,
    min_history=5,
)

df = apply_imputed_columns(df, df_imputed_only, impute_cols)

print(impute_report)
print(df[impute_cols].isna().sum())

                           column  NaNs_after_cat1  NaN_runs_after_cat1  \
0                          BA_Soc              829                    2   
1            BU_TotActPwr_Academy               87                    1   
2     BA_TotActPwr_BESS_AC_Panel1               87                    1   
3     BA_TotActPwr_BESS_AC_Panel2               87                    1   
4  BU_TotActPwr_SDB_EL_Substation             2355                    3   
5          BU_TotActPwr_Tech_Room               87                    1   
6                   PV_WS_AirTemp              433                    2   
7                 PV_WS_Radiation              433                    2   
8                    PV_WS_RelHum              433                    2   

   min_run  max_run  runs_cat2_(2h_to_24h)  runs_cat3_(>24h)  
0       87      742                      1                 1  
1       87       87                      1                 0  
2       87       87                      1                 0

In [44]:
ms = missing_summary(df)
print(ms[ms["missing_count"] > 0])

                                missing_count  missing_pct
BU_TotActPwr_SDB_EL_Substation            384         3.42


#### add calendar features

In [45]:
df = add_calendar_features(df, include_extended=True)
calendar_cols = get_calendar_feature_columns(include_extended=True)

##### Add lags ,roll , std, trend

In [46]:
target_cols = [
    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",
]

support_cols = [
    "BA_Soc",
    "PV_WS_AirTemp",
    "PV_WS_Radiation",
    "PV_WS_RelHum",
]

lags = [1, 2, 3, 4, 8, 12, 24, 48, 96, 192, 288, 672]
rolling_windows = [4, 12, 24, 48, 96, 288]
std_windows = [12, 24, 48, 96]
trend_pairs = [(1, 4), (1, 12), (1, 96), (96, 192)]

In [47]:
df = add_lag_features(df, target_cols, lags=lags)

df = add_rolling_features(
    df,
    target_cols,
    rolling_windows=rolling_windows,
    add_std_windows=std_windows,
)

df = add_trend_features(
    df,
    target_cols,
    trend_pairs=trend_pairs,
)


In [48]:
for target_col in target_cols:
    df = profile_features.add_day_ahead_history_features(
        df,
        target_col=target_col,
        freq_minutes=15,
    )

In [49]:
datasets = dataset_builder.build_multiple_target_datasets(
    df,
    target_cols=target_cols,
    support_cols=support_cols,
    calendar_cols=calendar_cols,
    include_calendar=True,
    include_lags=True,
    include_rolls=True,
    include_trends=True,
    include_history=True,
    drop_startup=True,
    startup_rows=672,
    drop_target_nan=False,
)

In [50]:
for name, dfx in datasets.items():
    print(name, dfx.shape)

BU_TotActPwr_Academy (10560, 50)
BA_TotActPwr_BESS_AC_Panel1 (10560, 50)
BA_TotActPwr_BESS_AC_Panel2 (10560, 50)
BU_TotActPwr_Tech_Room (10560, 50)
BU_TotActPwr_SDB_EL_Substation (10560, 50)


In [51]:
saved_files = dataset_builder.save_target_datasets_parquet(datasets, SAVE_DIR)

for k, v in saved_files.items():
    print(k, v, v.exists())

BU_TotActPwr_Academy C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min\df_BU_TotActPwr_Academy.parquet True
BA_TotActPwr_BESS_AC_Panel1 C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min\df_BA_TotActPwr_BESS_AC_Panel1.parquet True
BA_TotActPwr_BESS_AC_Panel2 C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min\df_BA_TotActPwr_BESS_AC_Panel2.parquet True
BU_TotActPwr_Tech_Room C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min\df_BU_TotActPwr_Tech_Room.parquet True
BU_TotActPwr_SDB_EL_Substation C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min\df_BU_TotActPwr_SDB_EL_Substation.parquet True


# Next Steps: Machine Learning Model Development


## 1. Time-Series Data Splitting
The dataset will be split chronologically into:
- **Training set**
- **Validation set**
- **Test set**

This avoids data leakage and ensures the model is evaluated on future unseen data.



## 2. Regression-Based Models
Since load forecasting is a continuous prediction task, regression models will be used, such as:
- Random Forest
- XGBoost


## 3. Multiple Load Forecasting
Separate models will be trained for different load signals while using a consistent training pipeline.



## 4. Model Evaluation
Performance will be evaluated using:
- **MAE**
- **RMSE**
- **MAPE**

## **without imputation and with imputation and comparision of XGboost**